# 01 · EDA: business entity resolution
Loads the raw TSVs with the same reader as the pipeline (`src/data.py`) and reproduces the numbers
used in the documentation: sizes, countries, match structure, noise patterns.

In [ ]:
import os, sys, re
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO); sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
from src.utils import load_config
from src.data import read_source, read_ground_truth, ground_truth_pairs
cfg = load_config()
src = {(sp, n): read_source(cfg, sp, n) for sp in ("train", "test") for n in (1, 2, 3)}
gt = read_ground_truth(cfg)
pd.DataFrame({f"{sp}_s{n}": [len(df)] for (sp, n), df in src.items()})

In [ ]:
# country labels per file (test adds France)
pd.DataFrame({f"{sp}_s{n}": df["country"].value_counts() for (sp, n), df in src.items()}).fillna(0).astype(int)

In [ ]:
# match structure: singletons, matches per S1, per source
pairs = ground_truth_pairs(cfg)
n = gt["matched_entity_ids"].map(lambda s: len([x for x in s.split(",") if x]))
print(f"singletons: {(n == 0).mean():.4f}  mean matches: {n.mean():.3f}  max: {n.max()}")
print("records matched to >1 S1:", int((pairs["rec_id"].value_counts() > 1).sum()))
n.clip(upper=10).value_counts().sort_index()

In [ ]:
# matched pairs always share the country label
c1 = src[("train", 1)].set_index("entity_id")["country"]
c23 = pd.concat([src[("train", 2)], src[("train", 3)]]).set_index("entity_id")["country"]
(pairs["s1_id"].map(c1).to_numpy() == pairs["rec_id"].map(c23).to_numpy()).mean()

In [ ]:
# share of S2/S3 records that match nothing (distractors), non-Latin names, empty addresses
s23 = pd.concat([src[("train", 2)], src[("train", 3)]], ignore_index=True)
matched = s23["entity_id"].isin(set(pairs["rec_id"]))
non_latin = ~s23["business_name"].map(str.isascii)
empty = s23["business_address"].str.strip() == ""
pd.DataFrame({"matched": [matched.mean()], "non_ascii_name": [non_latin.mean()],
              "empty_addr_matched": [empty[matched].mean()], "empty_addr_unmatched": [empty[~matched].mean()]})

In [ ]:
# a few matched groups
rng = np.random.default_rng(0)
s1 = src[("train", 1)].set_index("entity_id"); rec = s23.set_index("entity_id")
for sid in rng.choice(gt.loc[n >= 3, "source1_entity_id"].to_numpy(), 5, replace=False):
    r = s1.loc[sid]; print(f"{sid} | {r.business_name} | {r.business_address} | {r.country}")
    for m in pairs.loc[pairs.s1_id == sid, "rec_id"]:
        q = rec.loc[m]; print(f"    {m} | {q.business_name} | {q.business_address}")

In [ ]:
# last address component: state codes vs names vs native script, per source
for key in [("train", 1), ("train", 3), ("test", 2)]:
    a = src[key]["business_address"]
    print(key, a.str.rsplit(",", n=1).str[-1].str.strip().value_counts().head(12).to_dict())